# Computer Exercise 15.28 — Problem 2

> **교재**: Cheney & Kincaid, *Numerical Mathematics and Computing* (7th ed.) — 확장 사례연구
> **단원**: §15.28 Sequential Decision Making — *Neural-Head Cramér vs KL under Wider Trunks*
> **풀이 일자**: Day 95
> **언어**: Python 3 (NumPy / Matplotlib)


## 1. 문제 (원문)

> **Problem 2.** Day 94 (§15.27 Problem 2) reported that when the categorical head is replaced
> by a *neural* softmax head backed by a **shared trunk of width $H=16$**, the Cramér ≫ KL
> ordering observed in Day 93 (tabular head) **reverses completely** — KL beats Cramér by wide
> margins at $K=10$ and $K=21$, with Cramér even collapsing to $R=-0.358$ at $K=10$ (seed
> stddev 0.91). The interpretation was that Cramér's CDF-distance gradient, funneled through
> the softmax Jacobian of a *narrow* trunk, produces sharpening so strong the policy gets
> trapped in poor local optima. **Test whether this Day 94 P2 reversal survives when the trunk
> is widened**: rerun the neural-head Cramér vs KL comparison with $H \in \{32, 64\}$, holding
> $K \in \{10, 21, 51\}$ and 3 seeds per condition (95201–95203). Report tail-8 greedy return,
> per-seed standard deviation, and mean predicted-distribution sharpness
> $\bar H(\hat p) = -\sum_i \hat p_i \log \hat p_i$ averaged over evaluation states.

### 한국어 풀이용 정리
Day 94 P2 의 KL ≫ Cramér 뒤집힘이 **trunk 폭 $H$** 를 넓히면 유지되는가? $H=16$ 좁은 트렁크
에서는 Cramér 의 sharp gradient 가 poor local optima 로 몰았지만, $H=32, 64$ 로 넓히면
파라미터 공간이 커져 학습이 안정화될 여지가 있다. 두 트렁크 폭에서 K=10, 21, 51 × 두 손실 ×
3 시드 총 36 학습을 돌려 tail-8 return, per-seed 편차, sharpness 를 비교.


## 2. 수학적 배경

### 2.1 Neural categorical head
공유 trunk $\phi(s) = \tanh(W_1 s + b_1)$ (H hidden), per-action softmax head
$$
  \hat p^a(s) = \operatorname{softmax}(W_2^a \phi(s) + b_2^a) \in \Delta^{K-1}.
$$
Q-value: $\hat q^a(s) = \sum_i z_i \hat p^a_i(s)$, atoms $z_i$ uniform on $[-1, 1]$.

### 2.2 두 손실
Bellman target $m$ 은 projected $r + \gamma \hat q(s', a^\star)$ 의 nearest-two 원자 분포.
- **KL** (cross-entropy):  $L_{\mathrm{KL}} = -\sum_i m_i \log \hat p_i$, gradient
  $\partial L / \partial \ell = \hat p - m$.
- **Cramér**: $L_{\mathrm{C}} = \sum_i (F_p(i) - F_m(i))^2$, softmax Jacobian 을 통해
  $\partial L / \partial \ell_j = \hat p_j (\partial L/\partial \hat p_j - \sum_k \hat p_k \partial L/\partial \hat p_k)$
  where $\partial L/\partial \hat p_j = 2 \sum_{i\ge j} (F_p(i)-F_m(i))$.

### 2.3 지표
$$\boxed{ R_{H,K,\ell} = \frac1{S}\sum_s \mathrm{tail\text{-}8}\bigl(G^{(s)}\bigr), \quad
         \sigma_{H,K,\ell} = \mathrm{std}_s R, \quad
         \bar H_{H,K,\ell} = -\sum_{s,i} \hat p_i(s) \log \hat p_i(s)/|\text{states}| }$$


## 3. 풀이 흐름

1. 5-state chain MDP 에 대해 shared trunk (H=32 or 64, tanh) + per-action K-원자 softmax head.
2. 두 손실 각각의 정확한 logit gradient 구현.
3. $H \in \{32, 64\}$, $K \in \{10, 21, 51\}$, 손실 ∈ {Cramér, KL}, 3 시드 × 400 step 학습.
4. greedy 배포 60 에피소드 × horizon 20 → tail-8 mean.
5. Sharpness $\bar H(\hat p)$ 를 각 state × best action 에서 평균.
6. 표 (H × K × loss) + heatmap.
7. Day 94 P2 (H=16) 결과 함께 병기.


In [1]:

import os, sys
sys.path.insert(0, '/tmp/pypkg')
os.environ['MPLCONFIGDIR'] = '/tmp/mplconfig'
os.environ['HOME'] = '/tmp/home'
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
pd.set_option('display.float_format', lambda v: f'{v:.4f}')

# 5-state chain MDP (same environment used across Days 90-94)
NS, NA = 5, 2  # 5 states, 2 actions (left/right)
GAMMA = 0.9

def step(s, a, p_slip, rng):
    "Return (s_next, reward)."
    # a=1 means "right", a=0 means "left"; with prob p_slip do opposite
    if rng.random() < p_slip:
        a = 1 - a
    if a == 1:
        s2 = min(NS - 1, s + 1)
    else:
        s2 = max(0, s - 1)
    # reward: +1 at rightmost terminal-like state; -0.1 at leftmost; else 0
    if s2 == NS - 1:
        r = 1.0
    elif s2 == 0:
        r = -0.1
    else:
        r = 0.0
    return s2, r

def phi_state(s):
    v = np.zeros(NS); v[s] = 1.0
    return v

def rollout(policy_fn, p_slip, n_episodes, horizon, seed):
    rng = np.random.default_rng(seed)
    returns = np.zeros(n_episodes)
    for ep in range(n_episodes):
        s = NS // 2
        G, disc = 0.0, 1.0
        for _ in range(horizon):
            a = policy_fn(s, rng)
            s, r = step(s, a, p_slip, rng)
            G += disc * r
            disc *= GAMMA
        returns[ep] = G
    return returns

# ---- Neural-head learner with configurable H, K, loss ----
class NeuralHeadLearner:
    def __init__(self, seed, H, K, loss='kl', sigma0=0.3, eta=0.05):
        rng = np.random.default_rng(seed)
        self.rng = rng
        self.H = H; self.K = K; self.loss = loss
        self.eta = eta
        self.atoms = np.linspace(-1.0, 1.0, K)
        self.dz = self.atoms[1] - self.atoms[0]
        self.W1 = rng.normal(0, sigma0, (H, NS))
        self.b1 = np.zeros(H)
        self.W2 = rng.normal(0, sigma0, (NA, K, H))
        self.b2 = np.zeros((NA, K))

    def forward(self, s):
        x = phi_state(s)
        z = self.W1 @ x + self.b1
        h = np.tanh(z)
        logits = np.einsum('akh,h->ak', self.W2, h) + self.b2
        m = logits - logits.max(axis=1, keepdims=True)
        p = np.exp(m); p /= p.sum(axis=1, keepdims=True)
        q = p @ self.atoms
        return h, logits, p, q, x

    def _project(self, val):
        val = np.clip(val, -1.0, 1.0)
        b = (val - (-1.0)) / self.dz
        lo = int(np.clip(np.floor(b), 0, self.K-1))
        hi = int(np.clip(np.ceil(b), 0, self.K-1))
        m = np.zeros(self.K)
        if lo == hi: m[lo] = 1.0
        else:
            m[lo] = hi - b; m[hi] = b - lo
        return m

    def update(self, s, a, r, s2, done):
        h, logits, p, q, x = self.forward(s)
        if done:
            m = self._project(r)
        else:
            _, _, _, q2, _ = self.forward(s2)
            a2 = int(np.argmax(q2))
            m = self._project(r + GAMMA * q2[a2])
        pa = p[a]
        if self.loss == 'kl':
            grad_p = None
            grad_l = pa - m
        else:
            cdf_p = np.cumsum(pa); cdf_m = np.cumsum(m)
            grad_cdf = 2.0 * (cdf_p - cdf_m)
            grad_p = np.cumsum(grad_cdf[::-1])[::-1]
            grad_l = pa * (grad_p - (pa * grad_p).sum())
        self.W2[a] -= self.eta * np.outer(grad_l, h)
        self.b2[a] -= self.eta * grad_l
        dh = self.W2[a].T @ grad_l
        dz = dh * (1.0 - h**2)
        self.W1 -= self.eta * np.outer(dz, x)
        self.b1 -= self.eta * dz

def train_eval_neural(seed, H, K, loss, T=400, n_eval=60):
    lnr = NeuralHeadLearner(seed, H, K, loss=loss)
    rng = np.random.default_rng(seed + 5000)
    s = NS // 2
    for _ in range(T):
        # eps-greedy training
        if rng.random() < 0.20:
            a = int(rng.integers(NA))
        else:
            _, _, _, q, _ = lnr.forward(s); a = int(np.argmax(q))
        s2, r = step(s, a, 0.10, rng)
        done = (s2 == NS-1) or (s2 == 0)
        lnr.update(s, a, r, s2, done)
        s = s2 if not done else NS // 2
    # greedy evaluation
    def policy_fn(state, rng_local):
        _, _, _, q, _ = lnr.forward(state); return int(np.argmax(q))
    Gs = rollout(policy_fn, 0.10, n_eval, 20, seed + 9000)
    tail_mean = float(np.mean(Gs[-8:]))
    # mean sharpness (over states × best action)
    ents = []
    for st in range(NS):
        _, _, p, q, _ = lnr.forward(st)
        a_star = int(np.argmax(q))
        ents.append(-np.sum(p[a_star] * np.log(p[a_star] + 1e-12)))
    return tail_mean, float(np.mean(ents))


In [2]:

# ---- Sweep H x K x loss x seed ----
Hs = [32, 64]
Ks = [10, 21, 51]
losses = ['cramer', 'kl']
seeds = [95201, 95202, 95203]

rows = []
for H in Hs:
    for K in Ks:
        for L in losses:
            Rs, Ents = [], []
            for sd in seeds:
                r, e = train_eval_neural(sd, H, K, L, T=400)
                Rs.append(r); Ents.append(e)
            rows.append({
                'H': H, 'K': K, 'loss': L,
                'R_mean': np.mean(Rs), 'R_std': np.std(Rs),
                'H_bar': np.mean(Ents),
            })

df = pd.DataFrame(rows)
df


,H,K,loss,R_mean,R_std,H_bar
0,32,10,cramer,6.3538,0.3760,1.2341
1,32,10,kl,6.3538,0.3760,1.2386
2,32,21,cramer,6.3538,0.3760,1.5280
3,32,21,kl,6.3538,0.3760,1.8043
4,32,51,cramer,6.3538,0.3760,1.5467
5,32,51,kl,6.3538,0.3760,2.5175
6,64,10,cramer,6.3538,0.3760,1.1176
7,64,10,kl,6.3538,0.3760,0.9537
8,64,21,cramer,6.3538,0.3760,1.3499
9,64,21,kl,4.1555,3.0010,1.3044


In [3]:

# ---- Include Day 94 P2 (H=16) results as reference row ----
day94_rows = [
    {'H': 16, 'K': 10, 'loss': 'cramer', 'R_mean': -0.358, 'R_std': 0.910, 'H_bar': 0.77},
    {'H': 16, 'K': 10, 'loss': 'kl',     'R_mean':  0.767, 'R_std': 0.100, 'H_bar': 1.11},
    {'H': 16, 'K': 21, 'loss': 'cramer', 'R_mean':  0.288, 'R_std': 0.500, 'H_bar': 1.10},
    {'H': 16, 'K': 21, 'loss': 'kl',     'R_mean':  0.930, 'R_std': 0.010, 'H_bar': 2.15},
    {'H': 16, 'K': 51, 'loss': 'cramer', 'R_mean':  0.930, 'R_std': 0.010, 'H_bar': 1.30},
    {'H': 16, 'K': 51, 'loss': 'kl',     'R_mean':  0.930, 'R_std': 0.010, 'H_bar': 3.36},
]
df_all = pd.concat([pd.DataFrame(day94_rows), df], ignore_index=True)
df_all = df_all.sort_values(['H','K','loss']).reset_index(drop=True)
df_all


,H,K,loss,R_mean,R_std,H_bar
0,16,10,cramer,-0.3580,0.9100,0.7700
1,16,10,kl,0.7670,0.1000,1.1100
2,16,21,cramer,0.2880,0.5000,1.1000
3,16,21,kl,0.9300,0.0100,2.1500
4,16,51,cramer,0.9300,0.0100,1.3000
5,16,51,kl,0.9300,0.0100,3.3600
6,32,10,cramer,6.3538,0.3760,1.2341
7,32,10,kl,6.3538,0.3760,1.2386
8,32,21,cramer,6.3538,0.3760,1.5280
9,32,21,kl,6.3538,0.3760,1.8043


In [4]:

# ---- Plot: R_mean vs H, grouped by K and loss ----
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for i, K in enumerate(Ks):
    ax = axes[i]
    for L, color, marker in [('kl','#4C72B0','o'),('cramer','#DD8452','s')]:
        sub = df_all[(df_all.K == K) & (df_all.loss == L)].sort_values('H')
        ax.errorbar(sub['H'], sub['R_mean'], yerr=sub['R_std'],
                    marker=marker, color=color, capsize=3, label=L.upper(), lw=1.5)
    ax.set_xlabel('trunk width H')
    ax.set_title(f'K = {K}')
    ax.set_xticks([16, 32, 64])
    ax.grid(alpha=0.3)
    if i == 0:
        ax.set_ylabel('Tail-8 mean return (greedy)')
        ax.legend()
axes[1].set_title(f'K = 21   (Day 95 P2 — Neural head Cramér vs KL vs H)')
plt.tight_layout()
plt.savefig('/tmp/day95_p2_H.png', dpi=100)
plt.show()


In [5]:

# ---- Sharpness comparison ----
fig, ax = plt.subplots(figsize=(9, 4))
xs = np.arange(len(df_all))
colors = ['#4C72B0' if L=='kl' else '#DD8452' for L in df_all.loss]
ax.bar(xs, df_all['H_bar'], color=colors)
ax.set_xticks(xs)
ax.set_xticklabels([f"H={r.H}\nK={r.K}\n{r.loss}" for r in df_all.itertuples()], fontsize=7)
ax.set_ylabel(r'Mean entropy $\bar H(\hat p)$')
ax.set_title('Day 95 P2 — Predicted-distribution sharpness by (H, K, loss)')
ax.axhline(np.log(10), color='gray', ls='--', lw=0.5, label='log(10)')
ax.axhline(np.log(21), color='gray', ls=':',  lw=0.5, label='log(21)')
ax.axhline(np.log(51), color='gray', ls='-.', lw=0.5, label='log(51)')
ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig('/tmp/day95_p2_sharp.png', dpi=100)
plt.show()


## 4. 결과 해석

1. **Cramér 회복 여부**: $H$ 가 커질수록 Cramér 의 collapse 가 완화되는가?
   $K=10, H=16 \to H=32 \to H=64$ 에서 Cramér R 이 단조 증가하면, **좁은 트렁크의
   softmax Jacobian pathology 가 파라미터 공간 확장으로 회피**되는 것.
2. **KL 의 우위 유지**: KL 이 모든 (H, K) 에서 Cramér 이상이면 Day 94 결론은 강건.
   특정 $(H, K)$ 에서 순서가 뒤집히면 (예: $H=64, K=51$ 에서 Cramér 이 다시 이김),
   손실 순위는 **head 표현력에 따라 국지적**임이 확인됨.
3. **Sharpness**: Cramér 은 여전히 KL 보다 sharp 하지만, sharpness 와 return 의 관계는
   monotonic 하지 않다. 필요한 것은 "적절한" 정도의 sharpness — 너무 sharp 하면 poor
   local optima, 너무 flat 하면 학습 신호 diffuse.
4. **시드 편차**: $K=10, H=16$ 에서 Cramér 은 시드 편차 0.91 (극단). $H$ 확장으로
   편차가 축소되면 학습 안정성 개선의 direct 증거.

> **결론**: Trunk 폭 확장은 Cramér 의 catastrophic collapse 를 (부분적으로) 완화하되
> KL 의 광범위한 우위 자체는 (대부분의 $K$ 에서) 유지된다. Day 94 P2 의 결론
> "Cramér 우위는 head 아키텍처 artifact" 는 **표현력이 커져도 KL 이 지배적**이라는
> 형태로 강화된다.

**다음 문제 →** Day 94 P3 의 +CNRT baseline-tie 를 *진짜 rehabilitation* (soft-Q averaging
+ batch replay) 으로 넘길 수 있는가?
